In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [2]:
# 1. Create a Simple Dummy Dataset for Multimodal Learning
# We will simulate 10 houses with both Tabular data and an Image
class MultimodalHousingDataset(Dataset):
    def __init__(self, num_samples=10):
        # Tabular data: 3 features per house (e.g., Bedrooms, Bathrooms, Square Feet)
        self.tabular_data = torch.randn(num_samples, 3)
        
        # Image data: Simple 3-channel (RGB) images of size 64x64 pixels
        self.image_data = torch.randn(num_samples, 3, 64, 64)
        
        # Target data: The house price we want to predict (1 value per house)
        self.prices = torch.randn(num_samples, 1) * 100000 + 250000 # Simulated prices around $250k

    def __len__(self):
        return len(self.prices)

    def __getitem__(self, idx):
        return self.tabular_data[idx], self.image_data[idx], self.prices[idx]

# Instantiate our dataset and a standard data loader
dataset = MultimodalHousingDataset()
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [3]:
# 2. Build the Multimodal Network Architecture
class MultimodalHousingModel(nn.Module):
    def __init__(self):
        super(MultimodalHousingModel, self).__init__()
        
        # Branch 1: CNN to extract features from the house image
        self.image_branch = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1), # Output: 16 x 32 x 32
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # Output: 32 x 16 x 16
            nn.ReLU(),
            nn.Flatten(), # Flattens the image into a single 1D list of features
            nn.Linear(32 * 16 * 16, 8), # Reduce it down to 8 visual features
            nn.ReLU()
        )
        
        # Branch 2: Simple Linear Layer for the tabular data
        self.tabular_branch = nn.Sequential(
            nn.Linear(3, 8), # Convert 3 features (Bedrooms, etc.) into 8 tabular features
            nn.ReLU()
        )
        
        # Feature Fusion & Final Prediction
        # Glues the 8 image features + 8 tabular features together = 16 features total
        self.final_regressor = nn.Sequential(
            nn.Linear(8 + 8, 8),
            nn.ReLU(),
            nn.Linear(8, 1) # Output a single continuous number (The House Price)
        )

    def forward(self, tabular, image):
        # Pass the data through their separate branches
        img_features = self.image_branch(image)
        tab_features = self.tabular_branch(tabular)
        
        # Concatenate (glue) the visual features and tabular features side-by-side
        combined_features = torch.cat((img_features, tab_features), dim=1)
        
        # Predict the price using the combined features
        predicted_price = self.final_regressor(combined_features)
        return predicted_price

In [4]:
# 3. Initialize Model, Loss Function, and Optimizer
model = MultimodalHousingModel()
criterion = nn.MSELoss() # Mean Squared Error loss for regression task
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [5]:
# 4. Simple Training Loop
print("Starting Multimodal Model Training...")
model.train()

for epoch in range(5): # Train for 5 quick epochs for demonstration
    epoch_loss = 0.0
    for tabular, image, price in dataloader:
        # Clear the gradients from the previous step
        optimizer.zero_grad()
        
        # Forward pass: Feed both modalities into the model
        predictions = model(tabular, image)
        
        # Calculate loss
        loss = criterion(predictions, price)
        
        # Backward pass: Compute gradients and optimize model weights
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1}/5 - Loss: {epoch_loss/len(dataloader):.4f}")

print("\nMultimodal training script executed successfully!")

Starting Multimodal Model Training...
Epoch 1/5 - Loss: 97784873779.2000
Epoch 2/5 - Loss: 97351090995.2000
Epoch 3/5 - Loss: 95090088345.6000
Epoch 4/5 - Loss: 87062410854.4000
Epoch 5/5 - Loss: 65851313766.4000

Multimodal training script executed successfully!
